# Python bridge for 03b

## A short code refresher

Use this optional notebook alongside 03b for a closer look at the code behind cosine and Jaccard neighbor rankings. Plan for about 15–20 minutes. The example has four customers and four products, with all the data supplied below.

The 03b lecture explains the two meanings of similarity. This companion follows five code patterns:

1. build quantity and presence tables;
2. preserve a two-dimensional row for SciPy;
3. convert the returned distances to similarities;
4. attach customer labels and exclude the customer's own row; and
5. reuse the steps in a ranking function.

## Create a small purchase table

Each row gives one customer's total quantity for one product in a made-up set of purchases. Each customer-product pair appears once. An absent pair means that the customer did not buy that product in these records.

In [ ]:
import pandas as pd
from scipy.spatial.distance import cdist

In [ ]:
purchases = pd.DataFrame(
    {
        "customer_id": ["A", "A", "B", "B", "B", "C", "C", "D", "D"],
        "product": ["P1", "P2", "P1", "P2", "P3", "P1", "P2", "P3", "P4"],
        "quantity": [2, 1, 4, 2, 1, 1, 3, 1, 2],
    }
)

purchases

## Put one customer on each row

`pivot` places customer IDs on rows, products on columns, and quantities at their intersections. `.fillna(0)` fills the absent customer-product pairs with zero.

In [ ]:
quantities = purchases.pivot(
    index="customer_id", columns="product", values="quantity"
).fillna(0)

print("quantity-table shape:", quantities.shape)
quantities.astype(int)

The nine purchase rows have become a `(4, 4)` table: four customers described by the same four product columns. A's row is `[2, 1, 0, 0]`. The `.astype(int)` call displays whole-number quantities without changing the stored `quantities` table. `pivot` requires one value per customer-product pair, which is why the supplied records already contain totals.

## Record product presence

`.gt(0)` asks whether each quantity is positive. The result contains `True` for purchased products and `False` for products not purchased.

In [ ]:
presence = quantities.gt(0)

presence

A and C have the same presence row because both bought P1 and P2. Their quantities differ. Keep the Boolean table for Jaccard and the quantity table for cosine. Every customer here bought something, so each quantity row has a direction that cosine can compare.

## Select one customer for comparison

As in the 03a bridge, double brackets preserve a one-row DataFrame. `.to_numpy()` then extracts its numerical values. The selected row and comparison table must use the same product columns in the same order.

In [ ]:
customer_of_interest = "A"
selected_row = quantities.loc[[customer_of_interest]].to_numpy()
comparison_rows = quantities.to_numpy()

print("selected row:", selected_row)
print("selected shape:", selected_row.shape)
print("comparison shape:", comparison_rows.shape)

The shapes are `(1, 4)` and `(4, 4)`. Single brackets, `.loc[customer_of_interest]`, would produce a one-dimensional row that `cdist` cannot use as its first input.

## Read the returned distances

`cdist` compares each row of the first input with each row of the second. One customer compared with four customers produces one row of four distances.

In [ ]:
distance_array = cdist(selected_row, comparison_rows, metric="cosine")
distance_row = distance_array[0]

print("returned array:", distance_array.round(3))
print("returned shape:", distance_array.shape)
print("first row:", distance_row.round(3))
print("first-row shape:", distance_row.shape)

`[0]` selects the whole first row, changing the shape from `(1, 4)` to `(4,)`. It does not select just the first distance. The four positions still follow `quantities.index`: A, B, C, and D.

## Attach labels and rank other customers

For cosine and Jaccard, subtracting the returned distance from 1 gives the corresponding similarity. A labeled `Series` keeps each similarity attached to the customer it describes. This matters once we sort the values.

In [ ]:
cosine_similarities = pd.Series(
    1 - distance_row, index=quantities.index, name="cosine similarity"
)
cosine_ranking = cosine_similarities.drop(customer_of_interest).sort_values(
    ascending=False
)

cosine_ranking.round(3)

With A selected, B ranks first at about 0.976, followed by C at 0.707 and D at 0. `drop` removes A by its label before ranking. Removing a row merely because its similarity is 1 could also remove another customer with the same profile direction.

## Use the presence table for Jaccard

The call has the same shape, but both inputs now come from `presence` and the metric is `"jaccard"`. The labels are unchanged.

In [ ]:
jaccard_distances = cdist(
    presence.loc[[customer_of_interest]].to_numpy(),
    presence.to_numpy(),
    metric="jaccard",
)[0]
jaccard_similarities = pd.Series(
    1 - jaccard_distances, index=presence.index, name="Jaccard similarity"
)
jaccard_ranking = jaccard_similarities.drop(customer_of_interest).sort_values(
    ascending=False
)

jaccard_ranking.round(3)

For A, C now ranks first with similarity 1: both bought exactly P1 and P2. B shares those two products but also bought P3, giving similarity 2/3. Changing the representation and metric changes the ranking even though the source purchases stay the same.

## Collect the steps in a function

The function below accepts a table, a customer label, and one of the two metric names. It returns a labeled ranking. `.head(2)` then selects the first two other customers.

In [ ]:
def rank_neighbors(matrix, customer_id, metric):
    """Rank other customers using cosine or Jaccard similarity."""
    distances = cdist(
        matrix.loc[[customer_id]].to_numpy(),
        matrix.to_numpy(),
        metric=metric,
    )[0]
    similarities = pd.Series(1 - distances, index=matrix.index, name="similarity")
    return similarities.drop(customer_id).sort_values(ascending=False)

In [ ]:
cosine_neighbors = rank_neighbors(quantities, customer_of_interest, "cosine")
jaccard_neighbors = rank_neighbors(presence, customer_of_interest, "jaccard")

print("Cosine neighbors:")
print(cosine_neighbors.head(2).round(3))
print("Jaccard neighbors:")
print(jaccard_neighbors.head(2).round(3))

The function repeats the steps we just inspected. Its parameter `matrix` supplies both the selected row and the comparison rows, which keeps their columns aligned.

## Try one small modification

Change `customer_of_interest` from `"A"` to `"B"`, then rerun the selection and all later code cells. B should disappear from both returned rankings, while A becomes a candidate. Check that every displayed similarity stays attached to the customer from the corresponding comparison row.

## Ready for 03b

You are ready to return to 03b when you can recognize these patterns:

```text
records.pivot(index=..., columns=..., values=...).fillna(0)  # build a quantity table
quantities.gt(0)                                           # record product presence
matrix.loc[[customer_id]].to_numpy()                       # preserve one row
cdist(one_row, all_rows, metric=...)[0]                     # extract its distances
pd.Series(1 - distances, index=matrix.index)                # attach similarity labels
similarities.drop(customer_id).sort_values(ascending=False) # rank other customers
```

The lecture's ranking function returns a DataFrame with customer IDs and similarities in separate columns. It follows the same sequence. Look for the selected row, the matching input columns, the distance-to-similarity conversion, and the self-exclusion before reading the ranking.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).